In [5]:

from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np

df = pd.read_csv('data/spectral_feature_data.csv')
target_cols = [col for col in df.columns if col.startswith("p")]

# Select features: All columns that are NOT in the exclusion list
# This assumes your CSV contains: Spectral_Cols, ph, ec, Target_Cols, and ID
non_feature_cols = [col for col in df.columns if col.startswith('p4')]

feature_cols = [col for col in df.columns if col not in non_feature_cols]

#print(f"{target_cols}\n\n{non_feature_cols}\n\n{feature_cols}")
 
#input features
spectral_columns = [col for col in feature_cols if not col.startswith("p")]

In [6]:

results = []

# Assuming target_cols and spectral_columns are already defined
# Example: spectral_columns = ['410', '435', '460', '485', ...]

combinations = {"Spectral Only": [False, False],
                "Spectral + pH": [True, False],
                "Spectral + EC": [False, True],
                "Spectral + pH + EC": [True, True]}

imputer = SimpleImputer(strategy='mean')

for config_type in combinations.keys():
    print(f"\n========== Evaluating Config: {config_type} ==========")
    
    prediction_columns = spectral_columns.copy()
    
    maskpH = pd.Series(True, index=df.index)
    maskEC = pd.Series(True, index=df.index)

    if combinations[config_type][0]: # if we are to use ph value
        prediction_columns.append("p1.pH.index")
        maskpH = df["p1.pH.index"].notna()  
        
    if combinations[config_type][1]: # if we are to use EC value
        prediction_columns.append("p1.EC.ds_m")
        maskEC = df["p1.EC.ds_m"].notna()   

    # Update X for this specific configuration
    X = df[prediction_columns]
    
    # Combine the feature masks (rows where required features exist)
    feature_mask = maskpH & maskEC
    
    print(f"Features in use: {len(prediction_columns)} columns")
    
    for target in target_cols:
        
        if target not in df.columns:
            continue
            
        # Combine target mask with the feature masks properly
        target_mask = df[target].notna()
        final_mask = target_mask & feature_mask 
        
        y_clean = df.loc[final_mask, target]
        X_clean = X.loc[final_mask]
        
        if y_clean.shape[0] < 100:
            print(f"  -> Skipping {target}: Not enough data ({y_clean.shape[0]} rows).")
            continue
            
        # Handle NaNs in spectral columns to maintain parity with the PLSR test
        X_clean_imputed = imputer.fit_transform(X_clean)

        X_train, X_test, y_train, y_test = train_test_split(X_clean_imputed, y_clean, test_size=0.2, random_state=42)

        # Setup Grid Search for LightGBM
        param_grid = {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7]
        }
        
        # verbose=-1 prevents LightGBM from printing spammy warnings during the loop
        lgbm = LGBMRegressor(random_state=42, verbose=-1)
        grid_search = GridSearchCV(lgbm, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
        
        try:
            grid_search.fit(X_train, y_train)
        except Exception as e:
            print(f"  -> Error training {target}: {e}")
            continue
        
        best_lgbm = grid_search.best_estimator_
        
        y_pred = best_lgbm.predict(X_test)
        
        metrics = {
            'Config': config_type, 
            'Feature': target,
            # Store the dictionary of best parameters as a string for the CSV
            'Best_Params': str(grid_search.best_params_), 
            'R2': r2_score(y_test, y_pred),
            'MAE': mean_absolute_error(y_test, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            "MPE": np.mean((y_test - y_pred) / y_test) * 100
        }
        
        results.append(metrics)
        print(f"  -> Finished {target}: R2 = {metrics['R2']:.3f} | MAE = {metrics['MAE']:.3f}")

# Save and View Performance Table
if results:
    performance_df = pd.DataFrame(results)
    performance_df.to_csv('lgbm_performance_results_combinations.csv', index=False)
    print("\n--- LightGBM Performance Summary ---")
    print(performance_df)


========== Evaluating Config: Spectral Only ==========
Features in use: 18 columns


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  -> Finished p1.pH.index: R2 = 0.513 | MAE = 0.782


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  -> Finished p1.EC.ds_m: R2 = 0.161 | MAE = 0.167


...

c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  -> Finished p3.K.mg_kg: R2 = 0.210 | MAE = 204.366


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jagda\anaconda3\Lib\site-packages\numpy\_core\_methods.py:53: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


  -> Finished p3.P.mg_kg: R2 = 0.180 | MAE = 18.898
  -> Skipping p3.S.wt_pct: Not enough data (50 rows).
  -> Skipping p4.BD.g_cm3: Not enough data (50 rows).
  -> Skipping p4.CEC.cmolc_kg: Not enough data (50 rows).
  -> Finished p4.CF.wt_pct: R2 = 0.065 | MAE = 10.985
  -> Skipping p4.WR_10kPa.wt_pct: Not enough data (0 rows).
  -> Skipping p4.WR_1500kPa.wt_pct: Not enough data (0 rows).
  -> Skipping p4.WR_33kPa.wt_pct: Not enough data (0 rows).

--- LightGBM Performance Summary ---
                Config               Feature  \
0        Spectral Only           p1.pH.index   
1        Spectral Only            p1.EC.ds_m   
2        Spectral Only        p1.Clay.wt_pct   
3        Spectral Only        p1.Sand.wt_pct   
4        Spectral Only        p1.Silt.wt_pct   
5        Spectral Only           p2.N.wt_pct   
6        Spectral Only          p2.OC.wt_pct   
7        Spectral Only            p3.K.mg_kg   
8        Spectral Only            p3.P.mg_kg   
9        Spectral Only      

c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:
import optuna
from lightgbm import LGBMRegressor

def objective(trial, X_tr, X_te, y_tr, y_te):
    # Define the search space for LightGBM
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "verbose": -1
    }

    model = LGBMRegressor(**params, random_state=42)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    # We use MAE as the target to minimize for better field-readability
    return mean_absolute_error(y_te, preds)



results = []

# Assuming target_cols and spectral_columns are already defined
# Example: spectral_columns = ['410', '435', '460', '485', ...]

'''"Spectral Only": [False, False],
                "Spectral + pH": [True, False],
                "Spectral + EC": [False, True],'''

combinations = {"Spectral + pH + EC": [True, True]}


imputer = SimpleImputer(strategy='mean')

for config_type in combinations.keys():
    print(f"\n========== Evaluating Config: {config_type} ==========")
    
    prediction_columns = spectral_columns.copy()
    
    maskpH = pd.Series(True, index=df.index)
    maskEC = pd.Series(True, index=df.index)

    if combinations[config_type][0]: # if we are to use ph value
        prediction_columns.append("p1.pH.index")
        maskpH = df["p1.pH.index"].notna()  
        
    if combinations[config_type][1]: # if we are to use EC value
        prediction_columns.append("p1.EC.ds_m")
        maskEC = df["p1.EC.ds_m"].notna()   

    # Update X for this specific configuration
    X = df[prediction_columns]
    
    # Combine the feature masks (rows where required features exist)
    feature_mask = maskpH & maskEC
    
    print(f"Features in use: {len(prediction_columns)} columns")
    
    
# --- Inside your Council Loop ---
    for target in target_cols:
        # [Data cleaning and splitting logic here...]
        
        study = optuna.create_study(direction='minimize')
        # Use a lambda to pass the specific X/y train-test sets for this nutrient
        study.optimize(lambda trial: objective(trial, X_train, X_test, y_train, y_test), n_trials=50)

        best_lgbm = LGBMRegressor(**study.best_params, random_state=42, verbose=-1)
        best_lgbm.fit(X_train, y_train)
        
        y_pred = best_lgbm.predict(X_test)
        
        metrics = {
            'Config': config_type, 
            'Feature': target,
            # Store the dictionary of best parameters as a string for the CSV
            'Best_Params': str(grid_search.best_params_), 
            'R2': r2_score(y_test, y_pred),
            'MAE': mean_absolute_error(y_test, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            "MPE": np.mean((y_test - y_pred) / y_test) * 100
        }
        
        results.append(metrics)
        print(f"  -> Finished {target}: R2 = {metrics['R2']:.3f} | MAE = {metrics['MAE']:.3f}")

# Save and View Performance Table
if results:
    performance_df = pd.DataFrame(results)
    performance_df.to_csv('lgbm_performance_results_combinations(bayesian_search).csv', index=False)
    print("\n--- LightGBM Performance Summary ---")
    print(performance_df)



# --- Inside your Council Loop ---
for target in target_cols:
    # [Data cleaning and splitting logic here...]
    
    study = optuna.create_study(direction='minimize')
    # Use a lambda to pass the specific X/y train-test sets for this nutrient
    study.optimize(lambda trial: objective(trial, X_train, X_test, y_train, y_test), n_trials=50)

    best_lgbm = LGBMRegressor(**study.best_params, random_state=42, verbose=-1)
    best_lgbm.fit(X_train, y_train)

[I 2026-03-16 20:00:14,452] A new study created in memory with name: no-name-2e066ac3-c69a-4f93-86f6-db71bf9a4d56



========== Evaluating Config: Spectral Only ==========
Features in use: 18 columns


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:00:15,998] Trial 0 finished with value: 11.030759845164415 and parameters: {'n_estimators': 421, 'learning_rate': 0.04318504325416336, 'num_leaves': 142, 'max_depth': 12, 'min_child_samples': 29, 'reg_alpha': 0.0001089780177971533, 'reg_lambda': 0.0002345144238146366}. Best is trial 0 with value: 11.030759845164415.
c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:00:16,469] Trial 1 finished with value: 10.827201612185535 and parameters: {'n_estimators': 274, 'learning_rate': 0.04227691607691253, 'num_leaves': 100, 'max_depth': 7, 'min_child_samples': 50, 'reg_alpha': 0.13907460835101595, 'reg_lambda': 0.0005824310349646735

  -> Finished p1.pH.index: R2 = 0.091 | MAE = 10.780


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:00:58,748] Trial 0 finished with value: 10.84575461878861 and parameters: {'n_estimators': 107, 'learning_rate': 0.020301134247015216, 'num_leaves': 204, 'max_depth': 9, 'min_child_samples': 50, 'reg_alpha': 2.615315612779568, 'reg_lambda': 1.2548061990345998e-07}. Best is trial 0 with value: 10.84575461878861.
c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:00:59,249] Trial 1 finished with value: 11.306149173034251 and parameters: {'n_estimators': 243, 'learning_rate': 0.22728265989855906, 'num_leaves': 84, 'max_depth': 7, 'min_child_samples': 47, 'reg_alpha': 1.4052823754931697e-07, 'reg_lambda': 2.302444921418604}. Best

...

c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:26:09,627] Trial 1 finished with value: 11.09679783622459 and parameters: {'n_estimators': 322, 'learning_rate': 0.06316971353118962, 'num_leaves': 226, 'max_depth': 4, 'min_child_samples': 5, 'reg_alpha': 1.210384523943002e-07, 'reg_lambda': 0.11646326313307316}. Best is trial 0 with value: 10.984655389910982.
c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:26:09,965] Trial 2 finished with value: 10.907558058729187 and parameters: {'n_estimators': 210, 'learning_rate': 0.06503192675922036, 'num_leaves': 65, 'max_depth': 9, 'min_child_samples': 40, 'reg_alpha': 0.1870120112359232, 'reg_lambda': 2.761634100557924e-06}. Best

  -> Finished p4.WR_1500kPa.wt_pct: R2 = 0.093 | MAE = 10.769


c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:26:27,500] Trial 0 finished with value: 10.872683563241662 and parameters: {'n_estimators': 336, 'learning_rate': 0.03888899186152879, 'num_leaves': 43, 'max_depth': 9, 'min_child_samples': 38, 'reg_alpha': 0.2792619358668872, 'reg_lambda': 2.4684179732948928e-05}. Best is trial 0 with value: 10.872683563241662.
c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:26:27,689] Trial 1 finished with value: 11.02659317674516 and parameters: {'n_estimators': 133, 'learning_rate': 0.1457330335608922, 'num_leaves': 94, 'max_depth': 9, 'min_child_samples': 37, 'reg_alpha': 1.836988356498171e-08, 'reg_lambda': 0.0023093797607532424}. Be

  -> Finished p4.WR_33kPa.wt_pct: R2 = 0.089 | MAE = 10.768

--- LightGBM Performance Summary ---
                Config               Feature  \
0        Spectral Only           p1.pH.index   
1        Spectral Only            p1.EC.ds_m   
2        Spectral Only        p1.Clay.wt_pct   
3        Spectral Only        p1.Sand.wt_pct   
4        Spectral Only        p1.Silt.wt_pct   
..                 ...                   ...   
67  Spectral + pH + EC       p4.CEC.cmolc_kg   
68  Spectral + pH + EC          p4.CF.wt_pct   
69  Spectral + pH + EC    p4.WR_10kPa.wt_pct   
70  Spectral + pH + EC  p4.WR_1500kPa.wt_pct   
71  Spectral + pH + EC    p4.WR_33kPa.wt_pct   

                                          Best_Params        R2        MAE  \
0   {'learning_rate': 0.01, 'max_depth': 7, 'n_est...  0.091230  10.779706   
1   {'learning_rate': 0.01, 'max_depth': 7, 'n_est...  0.085128  10.775864   
2   {'learning_rate': 0.01, 'max_depth': 7, 'n_est...  0.091657  10.772312   
3   {'learnin

c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:26:45,354] Trial 0 finished with value: 10.813988932574976 and parameters: {'n_estimators': 207, 'learning_rate': 0.06327864342757916, 'num_leaves': 67, 'max_depth': 10, 'min_child_samples': 93, 'reg_alpha': 2.3745288248733747e-05, 'reg_lambda': 3.1514452942925747e-07}. Best is trial 0 with value: 10.813988932574976.
c:\Users\jagda\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-16 20:26:45,423] Trial 1 finished with value: 11.138118751777801 and parameters: {'n_estimators': 81, 'learning_rate': 0.015829120166625823, 'num_leaves': 48, 'max_depth': 3, 'min_child_samples': 44, 'reg_alpha': 0.004108405701963819, 'reg_lambda': 4.346194612160925}. 